In [1]:
# pandas: para leer y manipular los Excel como tablas (DataFrames)
import pandas as pd
import glob, os, unicodedata, time

In [12]:
CARPETA_ENTRADA = "."         
NOMBRE_HOJA_DATOS = "BDD_2024"
MESES_2024 = ["Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
ARCHIVO_SALIDA_2024 = "BDD_2024_AGOSTO_DICIEMBRE.xlsx"

In [13]:
#Quita tildes y pasa el texto a minúsculas
def normaliza(s):
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    return s.lower()

#Devuelve la lista de columnas que no tienen ningún dato
def columnas_totalmente_vacias(df):
    return df.columns[df.isna().all()].tolist()

#Busca, dentro de una lista de rutas de archivos
def encontrar_archivo_mes(mes, archivos):
    mes_norm = normaliza(mes)
    candidatos = [a for a in archivos if mes_norm in normaliza(os.path.basename(a))]
    if len(candidatos) == 0:
        return None  
    if len(candidatos) > 1:
        print(f"  Aviso: más de un archivo coincide con '{mes}': "
              f"{[os.path.basename(c) for c in candidatos]}. Se usará el primero.")
    return candidatos[0]

In [14]:
# Busca todos los .xlsx dentro de CARPETA_ENTRADA
archivos = [
    f for f in glob.glob(os.path.join(CARPETA_ENTRADA, "*.xlsx"))
    if not os.path.basename(f).startswith("~$")
]
print(f"Archivos encontrados en '{CARPETA_ENTRADA}': {len(archivos)}")
for a in sorted(archivos):
    print(" -", os.path.basename(a))

Archivos encontrados en '.': 5
 - 08._BDD_agosto_2024_SM_.xlsx
 - 09._BDD_septiembre_2024_final.xlsx
 - 10._BDD_octubre_2024_SM_.xlsx
 - 11. BDD_noviembre_2024_final_SM.xlsx
 - 12. BDD_diciembre_2024_final_SM.xlsx


In [ ]:
#Lee la hoja de datos de cada mes, las une todas en una sola tabla, limpia columnas vacías,
def unir_meses(meses, archivos, archivo_salida):

    print(f"\n=== Uniendo grupo: {archivo_salida} ===")
    dfs = []

    for mes in meses:
        archivo = encontrar_archivo_mes(mes, archivos)
        if archivo is None:
            print(f"  Aviso: no se encontró archivo para '{mes}', se omite.")
            continue

        t0 = time.time()
        try:
            df = pd.read_excel(archivo, sheet_name=NOMBRE_HOJA_DATOS)
        except ValueError:
            print(f"  Aviso: '{os.path.basename(archivo)}' no tiene una hoja llamada "
                  f"'{NOMBRE_HOJA_DATOS}', se omite.")
            continue

        df["MES"] = mes
        df["ARCHIVO_ORIGEN"] = os.path.basename(archivo)

        dfs.append(df)
        print(f"  {mes}: {os.path.basename(archivo)} -> "
              f"{df.shape[0]} filas x {df.shape[1]} columnas ({time.time()-t0:.1f}s)")

    if not dfs:
        print("  No se encontró ningún archivo para este grupo.")
        return None

    df_unificado = pd.concat(dfs, ignore_index=True, sort=False)

    vacias = columnas_totalmente_vacias(df_unificado)
    df_unificado = df_unificado.drop(columns=vacias)

    # Reordena para que MES y ARCHIVO_ORIGEN queden siempre al final,
    otras_columnas = [c for c in df_unificado.columns if c not in ["MES", "ARCHIVO_ORIGEN"]]
    df_unificado = df_unificado[otras_columnas + ["MES", "ARCHIVO_ORIGEN"]]

    print(f"  Total filas unidas: {df_unificado.shape[0]}")
    print(f"  Total columnas: {df_unificado.shape[1]} (se quitaron {len(vacias)} columnas 100% vacías)")

    t0 = time.time()
    df_unificado.to_excel(archivo_salida, sheet_name="BDD_UNIFICADO", index=False)
    print(f"  Archivo guardado en: {archivo_salida} ({time.time()-t0:.1f}s)")

    return df_unificado

In [16]:
#Agosto a Diciembre 2024
df_2024 = unir_meses(MESES_2024, archivos, ARCHIVO_SALIDA_2024)
df_2024["MES"].value_counts() if df_2024 is not None else None


=== Uniendo grupo: BDD_2024_AGOSTO_DICIEMBRE.xlsx ===
  Agosto: 08._BDD_agosto_2024_SM_.xlsx -> 13845 filas x 335 columnas (110.2s)
  Septiembre: 09._BDD_septiembre_2024_final.xlsx -> 15589 filas x 503 columnas (177.8s)
  Octubre: 10._BDD_octubre_2024_SM_.xlsx -> 17474 filas x 333 columnas (64.0s)
  Noviembre: 11. BDD_noviembre_2024_final_SM.xlsx -> 19293 filas x 333 columnas (64.6s)
  Diciembre: 12. BDD_diciembre_2024_final_SM.xlsx -> 21220 filas x 333 columnas (61.2s)
  Total filas unidas: 87421
  Total columnas: 383 (se quitaron 121 columnas 100% vacías)
  Archivo guardado en: BDD_2024_AGOSTO_DICIEMBRE.xlsx (1130.0s)


MES
Diciembre     21220
Noviembre     19293
Octubre       17474
Septiembre    15589
Agosto        13845
Name: count, dtype: int64